# PyWatermark Training on Colab

End-to-end Colab workflow: mount Drive, clone the repo, prepare a small COCO subset directly into Drive, and launch resumable training.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo and install dependencies

Replace `YOUR_REPO_URL` once, then rerun this cell whenever you start a new Colab session.

In [ ]:
import os
from pathlib import Path

REPO_URL = "YOUR_REPO_URL"
PROJECT_DIR = Path('/content/PyWatermark')

if PROJECT_DIR.exists():
    %cd /content/PyWatermark
    !git pull
else:
    %cd /content
    !git clone {REPO_URL} PyWatermark
    %cd /content/PyWatermark

!pip install -q -r requirements.txt

## 3. Prepare a lightweight COCO subset directly in Drive

This downloads `val2017` from the official COCO image zip and turns it into `train/val/test` folders under Drive. The counts below are a good starting point for Colab Free.

In [ ]:
%cd /content/PyWatermark
!python -m data.prepare_coco \
    --download-split val2017 \
    --raw-dir /content/drive/MyDrive/PyWatermark/raw \
    --output-root /content/drive/MyDrive/PyWatermark/datasets \
    --train-count 4000 \
    --val-count 500 \
    --test-count 500 \
    --force

## 4. Run a short smoke pass

Use this once before a longer training run to confirm that the dataset, checkpoints, and logging paths are all correct.

In [ ]:
%cd /content/PyWatermark
!python -m training.train \
    --train-dir /content/drive/MyDrive/PyWatermark/datasets/train \
    --val-dir /content/drive/MyDrive/PyWatermark/datasets/val \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint-dir /content/drive/MyDrive/PyWatermark/artifacts/checkpoints \
    --log-dir /content/drive/MyDrive/PyWatermark/artifacts/logs \
    --train-batch-size 8 \
    --eval-batch-size 8 \
    --num-workers 2 \
    --run-smoke-test

## 5. Clean-path debug training with attacks disabled

Run this before robust training when the model is stuck near random guessing. It removes all attacks so you can check whether the encoder and decoder can learn the basic watermarking task at all.

In [ ]:
%cd /content/PyWatermark
!python -m training.train \
    --train-dir /content/drive/MyDrive/PyWatermark/datasets/train \
    --val-dir /content/drive/MyDrive/PyWatermark/datasets/val \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint-dir /content/drive/MyDrive/PyWatermark/artifacts/checkpoints_clean \
    --log-dir /content/drive/MyDrive/PyWatermark/artifacts/logs_clean \
    --epochs 10 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --disable-attacks

## 6. Launch resumable robust training

Use this only after the clean-path debug run shows validation bit accuracy clearly above random. If Colab disconnects, reconnect and rerun from step 2 onward. `--auto-resume` will pick up from the latest checkpoint in Drive.

In [ ]:
%cd /content/PyWatermark
!python -m training.train \
    --train-dir /content/drive/MyDrive/PyWatermark/datasets/train \
    --val-dir /content/drive/MyDrive/PyWatermark/datasets/val \
    --test-dir /content/drive/MyDrive/PyWatermark/datasets/test \
    --checkpoint-dir /content/drive/MyDrive/PyWatermark/artifacts/checkpoints \
    --log-dir /content/drive/MyDrive/PyWatermark/artifacts/logs \
    --epochs 20 \
    --train-batch-size 16 \
    --eval-batch-size 16 \
    --num-workers 2 \
    --auto-resume